# Reuniones de TPM: fuentes, fechas y tabla por decisión

**Proyecto: ¿Podemos anticipar al Banco Central? · Período: 2010–2025**

Este notebook documenta y reproduce el procesamiento de la recopilación local
de reuniones de política monetaria. Cada fila final representa una decisión:
**sube, mantiene o baja**. No entrenamos modelos en esta etapa.

El resultado revisado contiene **161 decisiones**: 103 mantenciones, 31 bajas
y 27 subidas. Las fuentes fueron consultadas el **16 de octubre de 2026**.

**Qué se reproduce:** lectura de los insumos conservados, normalización de
fechas, correcciones documentadas, cruce con la TPM diaria y exportación.
La navegación y transcripción web originales no se vuelven a ejecutar.
Este notebook no descarga datos, no usa una API y no necesita ejecutar el
notebook `01_limpieza_y_procesamiento.ipynb`.

## 1. Organización y forma de ejecución

```text
data/raw/reuniones_tpm/            Fuentes y evidencia conservadas
data/raw/tpm_bcentral_diaria.xlsx  Excel original de niveles diarios
                  ↓
Normalizar fechas → alinear niveles → crear etiquetas → validar
                  ↓
data/processed/reuniones_tpm_2010_2025.csv
```

Seleccionar el kernel del entorno virtual del proyecto y ejecutar **todas las
celdas desde un kernel limpio**. Se usan las dependencias existentes: pandas,
openpyxl e IPython. El directorio de trabajo puede ser la raíz o `notebooks/`.

Las fuentes de `raw` se leen; la última celda de código regenera únicamente el
CSV de reuniones en `processed`. Los otros Excel y CSV no se modifican.

In [1]:
from pathlib import Path
import hashlib
import html
import json
import re
import sys

import pandas as pd
from IPython.display import display

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "src/processing/reuniones_tpm.py").is_file()
     and (p / "data/raw/reuniones_tpm").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Abre el notebook desde la raíz del proyecto o su carpeta notebooks/.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.processing.reuniones_tpm import construir, fecha_es, leer_fuentes

FUENTES_DIR = ROOT / "data/raw/reuniones_tpm"
TPM_PATH = ROOT / "data/raw/tpm_bcentral_diaria.xlsx"
SALIDA_PATH = ROOT / "data/processed/reuniones_tpm_2010_2025.csv"
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.max_rows", 20)
print(f"Raíz: {ROOT}")
print(f"Python: {sys.executable}")
print(f"pandas: {pd.__version__}")

Raíz: C:\Users\56974\Desktop\tpm-chile
Python: C:\Users\56974\Desktop\tpm-chile\.venv\Scripts\python.exe
pandas: 3.0.6


## 2. Inventario de fuentes y procedencia

Los JSON son **extractos y transcripciones de fuentes oficiales** elaborados
durante la recopilación; no son una base original descargada de una API.
Conservan URLs y, según la fuente, texto de la decisión, página, hash del PDF
y correcciones. El HTML conserva el anexo original de la Memoria 2014.

Calculamos los hashes de los insumos para identificar las copias utilizadas.
Los boletines PDF completos fueron una caché temporal fuera del repositorio;
no hacen falta para ejecutar este proceso local.

In [2]:
descripciones = {
    "rpm_boletines_2010_2012.json": "31 comunicados: 2010, 2011 y enero–julio de 2012",
    "rpm_fuentes_complementarias.json": "Fechas complementarias de 2012–2014 y referencias oficiales",
    "rpm_archivo_2015_2025.json": "Archivo por año, rangos y corrección de diciembre de 2023",
    "rpm_memoria_2014.html": "Anexo original con doce comunicados de 2014",
}
inventario = []
for nombre, contenido in descripciones.items():
    ruta = FUENTES_DIR / nombre
    if not ruta.is_file():
        raise FileNotFoundError(ruta)
    inventario.append({
        "archivo": nombre, "contenido": contenido,
        "bytes": ruta.stat().st_size,
        "sha256": hashlib.sha256(ruta.read_bytes()).hexdigest(),
    })
display(pd.DataFrame(inventario))

def leer_json(nombre):
    return json.loads((FUENTES_DIR / nombre).read_text(encoding="utf-8"))

boletines = leer_json("rpm_boletines_2010_2012.json")
complementarias = leer_json("rpm_fuentes_complementarias.json")
archivo = leer_json("rpm_archivo_2015_2025.json")
display(pd.DataFrame([
    {"fuente": "Boletines", "fecha_consulta": boletines["fecha_consulta"],
     "fechas": len(boletines["registros"])},
    {"fuente": "Complementarias", "fecha_consulta": complementarias["fecha_consulta"],
     "fechas": sum(len(g["fechas"]) for g in complementarias["grupos"])},
    {"fuente": "Archivo 2015–2025", "fecha_consulta": archivo["fecha_consulta"],
     "fechas": sum(len(f) for f in archivo["fechas_por_anio"].values())},
]))

,archivo,contenido,bytes,sha256
0,rpm_boletines_2010_2012.json,"31 comunicados: 2010, 2011 y enero–julio de 2012",18148,5f741e1f6eb2ccc2db1f5b232a8d272e0c721473727ee53810eabde9d1a33b47
1,rpm_fuentes_complementarias.json,Fechas complementarias de 2012–2014 y referencias oficiales,3249,6bc4ead050601bd4bd599d5378669aace90d379a5ff7e0ad77329100c553d77d
2,rpm_archivo_2015_2025.json,"Archivo por año, rangos y corrección de diciembre de 2023",3659,1b7e072fb8fdee3d5f891a01fc4077b850ff70d39406b11ebb5361853086c3ce
3,rpm_memoria_2014.html,Anexo original con doce comunicados de 2014,25710,7cb43d6fd575d633a1b15ab074341bbb6e01d7c8bc9dc165e01fc110aade0942


,fuente,fecha_consulta,fechas
0,Boletines,2026-10-16,31
1,Complementarias,2026-10-16,29
2,Archivo 2015–2025,2026-10-16,101


In [3]:
# Ejemplo de evidencia conservada: fecha, decisión y nivel anunciado.
display(pd.DataFrame(boletines["registros"])[[
    "fecha_texto", "decision_texto", "tpm_anunciada_pct", "pagina_pdf"
]].head(3))
print("URL del primer boletín:", boletines["registros"][0]["fuente_url"])

,fecha_texto,decision_texto,tpm_anunciada_pct,pagina_pdf
0,"Santiago, 14 de enero 2010","En su reunión mensual de política monetaria, el Consejo del Banco Central de Chile aco...",0.5,7
1,"Santiago, 11 de febrero de 2010","En su reunión mensual de política monetaria, el Consejo del Banco Central de Chile aco...",0.5,7
2,"Santiago, 18 de marzo de 2010","En su reunión mensual de política monetaria, el Consejo del Banco Central de Chile aco...",0.5,7


URL del primer boletín: https://si3.bcentral.cl/estadisticas/Principal1/informes/BOLETIN/pdf/BM012010.pdf


### Contraste del anexo HTML de 2014

La fuente complementaria transcribe doce fechas y niveles de la Memoria 2014.
Leemos también la copia HTML para comprobar esa transcripción. Extraemos solo
los encabezados de fecha y el primer párrafo de cada comunicado; no buscamos
porcentajes en el resto del texto económico, donde aparecen otras cifras.

In [4]:
contenido_html = (FUENTES_DIR / "rpm_memoria_2014.html").read_text(encoding="utf-8-sig")
pares = re.findall(r"<h[23][^>]*>(.*?)</h[23]>\s*<p[^>]*>(.*?)</p>",
                   contenido_html, flags=re.S | re.I)
registros_html = []
for encabezado, parrafo in pares:
    fecha_texto = html.unescape(re.sub(r"<[^>]+>", " ", encabezado)).strip()
    frase = " ".join(html.unescape(re.sub(r"<[^>]+>", " ", parrafo)).split())
    niveles = re.findall(r"(\d+(?:[,.]\d+)?)\s*%", frase)
    if "En su reunión" not in frase or not niveles:
        continue
    registros_html.append({
        "fecha_decision": fecha_es(fecha_texto, 2014),
        "tpm_anunciada_pct": float(niveles[-1].replace(",", ".")),
    })
memoria_2014 = pd.DataFrame(registros_html)
grupo_2014 = next(g for g in complementarias["grupos"] if g["anio"] == 2014)
transcripcion_2014 = pd.DataFrame({
    "fecha_decision": pd.to_datetime(grupo_2014["fechas"]),
    "tpm_anunciada_pct": grupo_2014["tpm_anunciadas_pct"],
})
pd.testing.assert_frame_equal(memoria_2014, transcripcion_2014, check_dtype=False)
display(memoria_2014)

,fecha_decision,tpm_anunciada_pct
0,2014-01-16,4.50
1,2014-02-18,4.25
2,2014-03-13,4.00
3,2014-04-17,4.00
4,2014-05-15,4.00
5,2014-06-12,4.00
6,2014-07-15,3.75
7,2014-08-14,3.50
8,2014-09-11,3.25
9,2014-10-16,3.00


## 3. Normalización de fechas y correcciones

La fecha objetivo es **la fecha de decisión/anuncio**. Para rangos de dos días
usamos el último: «31 de enero y 1 de febrero 2018» corresponde al 1 de febrero.
Si el texto no trae año, lo aporta el filtro anual de la fuente.

La función `leer_fuentes` centraliza esa lectura, conserva la referencia de
cada fila y aplica las correcciones documentadas. Usarla permite que el
notebook y el script de procesamiento sigan la misma regla.

Estas fechas no son un calendario de publicación de IPC, IMACEC o EEE.
La fecha efectiva de publicación de cada predictor se incorporará después.

In [5]:
ejemplos_texto = [
    ("31 de enero y 1 de febrero 2018", 2018),
    ("22 y 23 de Octubre de 2019", 2019),
    ("4 de diciembre", 2018),
]
display(pd.DataFrame([
    {"texto_fuente": texto, "anio_fuente": anio, "fecha_decision": fecha_es(texto, anio)}
    for texto, anio in ejemplos_texto
]))

fechas = leer_fuentes(ROOT)
display(fechas[["fecha_decision", "fecha_texto_fuente", "tipo_reunion"]].head(8))
print(f"Fechas únicas: {fechas['fecha_decision'].nunique()} / {len(fechas)} filas")

,texto_fuente,anio_fuente,fecha_decision
0,31 de enero y 1 de febrero 2018,2018,2018-02-01
1,22 y 23 de Octubre de 2019,2019,2019-10-23
2,4 de diciembre,2018,2018-12-04


,fecha_decision,fecha_texto_fuente,tipo_reunion
0,2010-01-14,"Santiago, 14 de enero 2010",ordinaria
1,2010-02-11,"Santiago, 11 de febrero de 2010",ordinaria
2,2010-03-18,"Santiago, 18 de marzo de 2010",ordinaria
3,2010-04-15,"Santiago, 15 de abril de 2010",ordinaria
4,2010-05-13,"Santiago, 13 de mayo de 2010",ordinaria
5,2010-06-15,"Santiago, 15 de junio de 2010",ordinaria
6,2010-07-15,"Santiago, 15 de julio de 2010",ordinaria
7,2010-08-12,"Santiago, 12 de agosto de 2010",ordinaria


Fechas únicas: 161 / 161 filas


### Corrección de diciembre de 2023 y cierre de 2025

El archivo web listaba el **20 de diciembre de 2023**; el comunicado y la
minuta de la sesión 302 indican el **19**. Conservamos el texto original y
la corrección con su fuente. El nuevo nivel aparece en la serie diaria el 20:
eso no cambia la fecha del anuncio.

Además, el filtro 2025 devolvía siete fechas; la reunión del 16 de diciembre
aparecía en la tabla inicial de 2026. Se incorporó y contrastó documentalmente.
Por eso no basta contar las filas de un único filtro del sitio.

In [6]:
display(pd.DataFrame(archivo["correcciones_verificadas"])[[
    "fecha_listado", "fecha_decision", "tpm_anunciada_pct", "fuente_url"
]])
display(fechas.loc[fechas["fecha_decision"].isin(pd.to_datetime([
    "2023-12-19", "2025-12-16"
]))][["fecha_decision", "fecha_texto_fuente", "tpm_anunciada_verificada_pct"]])

,fecha_listado,fecha_decision,tpm_anunciada_pct,fuente_url
0,2023-12-20,2023-12-19,8.25,https://www.bcentral.cl/web/banco-central/contenido/-/detalle/comunicado-reunion-de-po...


,fecha_decision,fecha_texto_fuente,tpm_anunciada_verificada_pct
144,2023-12-19,20 de diciembre 2023,8.25
160,2025-12-16,16 de diciembre 2025,4.50


## 4. Leer la TPM diaria original

Leemos la hoja `Cuadro` con encabezados en la tercera fila (`header=2`),
usando las dos primeras columnas. Las fechas ya son valores de Excel; los
niveles son porcentajes: **0.5 significa 0,50%**.

Se conserva toda la historia del archivo para poder encontrar la observación
previa o posterior a una decisión. No usamos el CSV de limpieza del equipo.
La serie diaria sirve para recuperar niveles; las fechas de reunión provienen
de las fuentes del apartado anterior.

In [7]:
tpm_diaria = pd.read_excel(TPM_PATH, sheet_name="Cuadro", header=2)
tpm_diaria = tpm_diaria.iloc[:, :2].set_axis(["fecha", "tpm_pct"], axis=1)
tpm_diaria["fecha"] = pd.to_datetime(tpm_diaria["fecha"], errors="raise")
tpm_diaria["tpm_pct"] = pd.to_numeric(tpm_diaria["tpm_pct"], errors="raise")
tpm_diaria = tpm_diaria.sort_values("fecha").reset_index(drop=True)
assert not tpm_diaria.isna().any().any(), "Hay datos faltantes en la TPM diaria"
assert not tpm_diaria["fecha"].duplicated().any(), "Hay fechas diarias duplicadas"
print(f"Serie diaria: {len(tpm_diaria):,} filas; "
      f"{tpm_diaria['fecha'].min().date()} a {tpm_diaria['fecha'].max().date()}")
display(tpm_diaria.loc[tpm_diaria["fecha"].between("2010-06-14", "2010-06-18")])

Serie diaria: 7,391 filas; 1997-02-07 a 2026-09-22


,fecha,tpm_pct
3338,2010-06-14,0.5
3339,2010-06-15,0.5
3340,2010-06-16,1.0
3341,2010-06-17,1.0
3342,2010-06-18,1.0


## 5. Alinear el corte y el resultado de cada reunión

La regla acordada es predecir **al cierre del día hábil anterior a la decisión**,
en `America/Santiago`. En esta versión usamos como calendario operativo las
fechas presentes en la TPM diaria: el corte es la última observación
**estrictamente anterior** al anuncio. Así no suponemos que todo lunes–viernes
sea hábil. Falta contrastar este calendario con uno bancario independiente.

Para recuperar el nivel posterior tomamos la primera observación
**estrictamente posterior** al anuncio. Esa fecha no es necesariamente la
fecha de decisión ni certifica por sí sola la entrada en vigencia legal.

`merge_asof` realiza ambos cruces respetando el orden temporal:

| Cruce | Dirección | Coincidencia en la misma fecha |
|---|---|---|
| Corte y TPM previa | Hacia atrás | Excluida |
| Observación y TPM posterior | Hacia adelante | Excluida |

En una reunión de dos días, el corte puede coincidir con el primer día de
la reunión, porque predecimos antes de la decisión, no antes de su inicio.

In [8]:
previa = tpm_diaria.rename(columns={"fecha": "fecha_corte", "tpm_pct": "tpm_previa_pct"})
posterior = tpm_diaria.rename(columns={
    "fecha": "fecha_observacion_posterior", "tpm_pct": "tpm_posterior_pct"
})
alineadas = pd.merge_asof(
    fechas, previa, left_on="fecha_decision", right_on="fecha_corte",
    direction="backward", allow_exact_matches=False,
)
alineadas = pd.merge_asof(
    alineadas, posterior, left_on="fecha_decision", right_on="fecha_observacion_posterior",
    direction="forward", allow_exact_matches=False,
)
display(alineadas[["fecha_decision", "fecha_corte", "tpm_previa_pct",
                   "fecha_observacion_posterior", "tpm_posterior_pct"]].head(6))

,fecha_decision,fecha_corte,tpm_previa_pct,fecha_observacion_posterior,tpm_posterior_pct
0,2010-01-14,2010-01-13,0.5,2010-01-15,0.5
1,2010-02-11,2010-02-10,0.5,2010-02-12,0.5
2,2010-03-18,2010-03-17,0.5,2010-03-19,0.5
3,2010-04-15,2010-04-14,0.5,2010-04-16,0.5
4,2010-05-13,2010-05-12,0.5,2010-05-14,0.5
5,2010-06-15,2010-06-14,0.5,2010-06-16,1.0


## 6. Construir la etiqueta y ejecutar las validaciones

```text
variacion_pb = (TPM posterior − TPM previa) × 100
variacion_pb > 0 → sube;  = 0 → mantiene;  < 0 → baja
```

Pasar de 0,50% a 1,00% es subir **50 puntos base**. Una fila de mantención
es una reunión identificada en las fuentes, no cualquier día sin cambio.

Después de mostrar el cálculo, ejecutamos `construir`, la implementación
canónica utilizada también desde la terminal. Esta valida duplicados,
observaciones anteriores/posteriores, discrepancias con niveles documentados
y la correspondencia única entre cada cambio diario y una reunión. Si alguna
validación falla, no llegamos a la celda que exporta el CSV.

In [9]:
alineadas["variacion_pb"] = (
    (alineadas["tpm_posterior_pct"] - alineadas["tpm_previa_pct"]) * 100
).round(6)
alineadas["decision"] = alineadas["variacion_pb"].map(
    lambda cambio: "sube" if cambio > 0 else "baja" if cambio < 0 else "mantiene"
)

reuniones = construir(ROOT)
columnas_calculo = ["fecha_decision", "fecha_corte", "tpm_previa_pct",
                    "fecha_observacion_posterior", "tpm_posterior_pct",
                    "variacion_pb", "decision"]
pd.testing.assert_frame_equal(alineadas[columnas_calculo], reuniones[columnas_calculo])

documentadas = reuniones["tpm_anunciada_verificada_pct"].notna()
diferencias_documentadas = (
    reuniones.loc[documentadas, "tpm_posterior_pct"]
    - reuniones.loc[documentadas, "tpm_anunciada_verificada_pct"]
).abs()
cambios_diarios = tpm_diaria["tpm_pct"].diff().abs().gt(1e-8)
en_periodo = tpm_diaria["fecha"].between("2010-01-01", "2025-12-31")
controles = pd.Series({
    "161 reuniones en el período revisado": len(reuniones) == 161,
    "Fechas de reunión sin duplicados": not reuniones["fecha_decision"].duplicated().any(),
    "Cortes estrictamente anteriores al anuncio": reuniones["fecha_corte"].lt(reuniones["fecha_decision"]).all(),
    "Observaciones posteriores al anuncio": reuniones["fecha_observacion_posterior"].gt(reuniones["fecha_decision"]).all(),
    "Niveles documentados coinciden": diferencias_documentadas.lt(1e-8).all(),
    "58 cambios diarios y 58 decisiones de cambio": (
        int((cambios_diarios & en_periodo).sum()) == 58
        and int(reuniones["decision"].ne("mantiene").sum()) == 58
    ),
    "Demostración coincide con el procesamiento canónico": True,
}, name="cumple")
assert controles.all(), "Hay controles incumplidos; revisar antes de exportar"
display(controles.to_frame())

,cumple
161 reuniones en el período revisado,True
Fechas de reunión sin duplicados,True
Cortes estrictamente anteriores al anuncio,True
Observaciones posteriores al anuncio,True
Niveles documentados coinciden,True
58 cambios diarios y 58 decisiones de cambio,True
Demostración coincide con el procesamiento canónico,True


### Alcance del contraste documental

Las 161 fechas tienen referencias oficiales. **48 niveles de TPM** fueron
contrastados además con el texto del documento; para las otras **113 filas**
el nivel posterior se obtiene de la serie diaria original. Los vacíos de
`tpm_anunciada_verificada_pct` indican falta de ese contraste adicional,
no ausencia de una reunión o de su nivel en la tabla final.

In [10]:
display(reuniones["contraste_tpm"].value_counts().rename_axis("contraste").to_frame("reuniones"))
casos = pd.to_datetime([
    "2010-01-14", "2010-04-15", "2010-05-13", "2010-06-15",
    "2014-04-17", "2018-02-01", "2020-03-16", "2023-12-19",
])
display(reuniones.loc[reuniones["fecha_decision"].isin(casos),
                     columnas_calculo + ["tipo_reunion"]])

,reuniones
contraste,
derivada_de_serie_diaria,113
coincide_con_documento,48


,fecha_decision,fecha_corte,tpm_previa_pct,fecha_observacion_posterior,tpm_posterior_pct,variacion_pb,decision,tipo_reunion
0,2010-01-14,2010-01-13,0.50,2010-01-15,0.50,0.0,mantiene,ordinaria
3,2010-04-15,2010-04-14,0.50,2010-04-16,0.50,0.0,mantiene,ordinaria
4,2010-05-13,2010-05-12,0.50,2010-05-14,0.50,0.0,mantiene,ordinaria
5,2010-06-15,2010-06-14,0.50,2010-06-16,1.00,50.0,sube,ordinaria
51,2014-04-17,2014-04-16,4.00,2014-04-21,4.00,0.0,mantiene,ordinaria
96,2018-02-01,2018-01-31,2.50,2018-02-02,2.50,0.0,mantiene,ordinaria
113,2020-03-16,2020-03-13,1.75,2020-03-17,1.00,-75.0,baja,extraordinaria
144,2023-12-19,2023-12-18,9.00,2023-12-20,8.25,-75.0,baja,ordinaria


## 7. Revisar cobertura y distribución

En la muestra inicial de revisión se saltó de marzo a junio de 2010 para
mostrar una subida. Las reuniones de **15 de abril y 13 de mayo** sí existían
y mantuvieron la TPM en 0,50%; están incluidas aquí.

La frecuencia ordinaria pasa de 12 a 8 reuniones por año desde 2018. En 2020
se agrega la extraordinaria del 16 de marzo. Esta fecha se conoce
retrospectivamente: conservarla no significa que la convocatoria se conociera
al cierre del día anterior. Su uso en el experimento predictivo se decidirá
antes de entrenar modelos.

In [11]:
resumen_anual = pd.crosstab(reuniones["fecha_decision"].dt.year, reuniones["decision"])
resumen_anual = resumen_anual.reindex(columns=["baja", "mantiene", "sube"], fill_value=0)
resumen_anual["total"] = resumen_anual.sum(axis=1)
resumen_anual.index.name = "anio"
display(resumen_anual)
display(reuniones["decision"].value_counts().rename_axis("decision").to_frame("reuniones"))
display(reuniones.loc[reuniones["tipo_reunion"].eq("extraordinaria"),
                     columnas_calculo + ["tipo_reunion"]])

decision,baja,mantiene,sube,total
anio,,,,
2010,0,5,7,12
2011,0,7,5,12
2012,1,11,0,12
2013,2,10,0,12
2014,6,6,0,12
2015,0,10,2,12
2016,0,12,0,12
2017,4,8,0,12
2018,0,7,1,8


,reuniones
decision,
mantiene,103
baja,31
sube,27


,fecha_decision,fecha_corte,tpm_previa_pct,fecha_observacion_posterior,tpm_posterior_pct,variacion_pb,decision,tipo_reunion
113,2020-03-16,2020-03-13,1.75,2020-03-17,1.0,-75.0,baja,extraordinaria


## 8. Guardar y comprobar la tabla reproducible

La salida se guarda en `data/processed/reuniones_tpm_2010_2025.csv`, con fechas
ISO `AAAA-MM-DD`, niveles numéricos y enlaces de procedencia. Se regenera
desde los insumos locales; no depende de un CSV generado en una ejecución
anterior. Volvemos a leer el archivo para verificar todas sus columnas.

In [12]:
SALIDA_PATH.parent.mkdir(parents=True, exist_ok=True)
reuniones.to_csv(SALIDA_PATH, index=False, encoding="utf-8", date_format="%Y-%m-%d")

guardada = pd.read_csv(SALIDA_PATH)
esperada = reuniones.copy()
columnas_fecha = ["fecha_decision", "fecha_corte", "fecha_observacion_posterior"]
for columna in columnas_fecha:
    esperada[columna] = esperada[columna].dt.strftime("%Y-%m-%d")
# CSV representa textos vacíos como campos vacíos, leídos por defecto como NaN.
for columna in esperada.select_dtypes(include=["object", "str"]).columns:
    esperada[columna] = esperada[columna].replace("", float("nan"))
pd.testing.assert_frame_equal(guardada, esperada, check_dtype=False)
print(f"CSV verificado: {SALIDA_PATH}")
print(f"Filas: {len(guardada)} | Columnas: {len(guardada.columns)}")
print("SHA256:", hashlib.sha256(SALIDA_PATH.read_bytes()).hexdigest())
display(guardada[["fecha_decision", "fecha_corte", "decision"]].head(6))

CSV verificado: C:\Users\56974\Desktop\tpm-chile\data\processed\reuniones_tpm_2010_2025.csv
Filas: 161 | Columnas: 15
SHA256: 6dd97115161df71aec94738cabc7dc81bba098864166701a459fc83a45dba264


,fecha_decision,fecha_corte,decision
0,2010-01-14,2010-01-13,mantiene
1,2010-02-11,2010-02-10,mantiene
2,2010-03-18,2010-03-17,mantiene
3,2010-04-15,2010-04-14,mantiene
4,2010-05-13,2010-05-12,mantiene
5,2010-06-15,2010-06-14,sube


## 9. Qué falta antes del modelamiento

- Integrar IPC, IMACEC, desempleo, dólar, cobre y eventualmente EEE, usando
  **fechas efectivas de publicación no posteriores al corte**. El mes al que
  corresponde un indicador no equivale al día en que se publicó.
- Tratar o declarar las revisiones históricas de los indicadores y comprobar
  el calendario bancario utilizado para el corte.
- Decidir el tratamiento de la reunión extraordinaria y de los períodos con
  cobertura incompleta de predictores. El tamaño final puede ser menor a 161.
- Reservar `tpm_posterior_pct`, `variacion_pb`, `decision` y la fecha de la
  observación posterior para construir/evaluar el resultado: **no son
  predictores disponibles antes de la decisión**. El nivel anunciado y su
  contraste documental tampoco deben introducir información del resultado.

El diccionario completo, las fuentes y las decisiones metodológicas están
en [docs/reuniones_tpm.md](../docs/reuniones_tpm.md). La carpeta de insumos
tiene su [propio inventario](../data/raw/reuniones_tpm/README.md).

El mismo procesamiento validado se puede ejecutar desde la raíz con
`python -m src.processing.reuniones_tpm`. Las pruebas del módulo se ejecutan
con `python -m unittest discover -s tests -v`.